In [ ]:
import requests

PEXELS_API_KEY = "OzA5NAaBNk7otiMnUDvQS4ZwodwYraXZHHzw5FDV05WxGZTO6T1UmJN3"
url = "https://api.pexels.com/v1/search?query=construction"

headers = {"Authorization": PEXELS_API_KEY}
response = requests.get(url, headers=headers)

# Extract rate limit headers
rate_limit = response.headers.get("X-Ratelimit-Limit")
remaining = response.headers.get("X-Ratelimit-Remaining")
reset_time = response.headers.get("X-Ratelimit-Reset")

print(f"Rate Limit: {rate_limit}, Remaining: {remaining}, Reset Time: {reset_time}")


In [ ]:
import os

# 📂 Set transfer learning folder
base_path = r"E:\yolo_project\transfer_learning"
os.makedirs(base_path, exist_ok=True)

# 🔄 Create dataset folders (images + labels)
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(base_path, split, "images"), exist_ok=True)
    os.makedirs(os.path.join(base_path, split, "labels"), exist_ok=True)

print("✅ YOLO dataset folders created successfully!")


In [ ]:
import torch.nn as nn
import torch

class FeatureFusion(nn.Module):
    def __init__(self):
        super(FeatureFusion, self).__init__()
        self.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1)
        self.pool = nn.MaxPool2d(2, 2)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))
        x = self.pool(torch.relu(self.conv2(x)))
        return x

fusion_model = FeatureFusion()
print(fusion_model)


This would extract additional spatial features, improving detection accuracy when merged with YOLO.

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, in_dim):
        super(SelfAttention, self).__init__()
        self.query = nn.Conv2d(in_dim, in_dim // 8, 1)
        self.key = nn.Conv2d(in_dim, in_dim // 8, 1)
        self.value = nn.Conv2d(in_dim, in_dim, 1)

    def forward(self, x):
        Q = self.query(x)
        K = self.key(x)
        V = self.value(x)
        attention = torch.softmax(Q @ K.transpose(-2, -1), dim=-1)
        x = attention @ V
        return x

attention_module = SelfAttention(in_dim=128)
print(attention_module)


This prioritizes small objects like helmets, cables, and scaffolding while filtering irrelevant clutter.

In [ ]:
import torch
from ultralytics import YOLO

# Ensure GPU usage
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.cuda.set_device(0)

# Load YOLO11 pretrained model
base_model = YOLO("yolo11n.pt").to(device)
print("✅ YOLO11 model loaded successfully.")


In [ ]:
class EnhancedYOLO11(nn.Module):
    def __init__(self, base_model):
        super(EnhancedYOLO11, self).__init__()
        self.base_model = base_model
        self.fusion = FeatureFusion()
        self.attention = SelfAttention(in_dim=128)
        
    def forward(self, x):
        yolo_features = self.base_model(x)
        fused_features = self.fusion(yolo_features)
        enhanced_features = self.attention(fused_features)
        return enhanced_features

enhanced_model = EnhancedYOLO11(base_model).to(device)
print("✅ Enhanced YOLO11 model ready.")


In [ ]:
import os
import json

# 📂 Paths
dataset_path = r"E:\yolo_project\mined"  # Existing dataset
labels_path = r"E:\yolo_project\transfer_learning\labels"  # New annotation folder
os.makedirs(labels_path, exist_ok=True)

# 🔄 Convert bounding box annotations to YOLO format
def convert_annotation(image_name, category, split):
    json_file = image_name.replace(".jpg", ".json").replace(".png", ".json")
    json_path = os.path.join(dataset_path, split, category, json_file)

    if not os.path.exists(json_path):
        print(f"⚠️ No annotation file found for {image_name}. Skipping...")
        return None

    with open(json_path, "r") as f:
        data = json.load(f)

    img_width = data.get("imageWidth", 640)
    img_height = data.get("imageHeight", 640)

    yolo_annotations = []
    for obj in data["shapes"]:
        class_id = obj["label"]  # Ensure labels match class IDs
        x_min, y_min, x_max, y_max = obj["points"][0][0], obj["points"][0][1], obj["points"][1][0], obj["points"][1][1]

        # Convert bounding box to YOLO format
        x_center = ((x_min + x_max) / 2) / img_width
        y_center = ((y_min + y_max) / 2) / img_height
        width = (x_max - x_min) / img_width
        height = (y_max - y_min) / img_height

        yolo_annotations.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return yolo_annotations

# 🔄 Process images
for split in ["train", "val", "test"]:
    split_path = os.path.join(dataset_path, split)
    if not os.path.exists(split_path):
        continue

    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if not os.path.isdir(category_path):
            continue

        for image_file in os.listdir(category_path):
            if image_file.endswith((".jpg", ".png")):
                annotations = convert_annotation(image_file, category, split)
                
                if annotations:
                    label_filename = image_file.replace(".jpg", ".txt").replace(".png", ".txt")
                    with open(os.path.join(labels_path, label_filename), "w") as f:
                        f.write("\n".join(annotations))

print("✅ YOLO annotations saved in E:\\yolo_project\\transfer_learning\\labels!")


In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image

# 📂 Base path for mined dataset
base_path = r"E:\yolo_project\mined"
transfer_path = r"E:\yolo_project\transfer_learning\labels"  # ✅ Save labels here
os.makedirs(transfer_path, exist_ok=True)

# ✅ Define dataset loader
class YOLODataset(Dataset):
    def __init__(self, dataset_split, transform=None, save_labels=False):
        self.split_path = os.path.join(base_path, dataset_split)
        self.transform = transform
        self.save_labels = save_labels
        self.label_save_path = os.path.join(transfer_path, dataset_split)  # ✅ Save labels in transfer_learning
        os.makedirs(self.label_save_path, exist_ok=True)

        # 🔄 Collect all images + dynamically assign labels
        self.img_files = []
        self.labels = []
        for category in os.listdir(self.split_path):
            category_path = os.path.join(self.split_path, category)
            if not os.path.isdir(category_path):
                continue

            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(category)  # ✅ Use folder name as label

        print(f"📊 Found {len(self.img_files)} images in {dataset_split}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        category_label = self.labels[idx]  # ✅ Folder name as label

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # ✅ Save label as YOLO format
        label_file = os.path.join(self.label_save_path, os.path.basename(img_path).replace(".jpg", ".txt").replace(".png", ".txt"))
        if self.save_labels:
            with open(label_file, "w") as f:
                f.write(f"{category_label} 0.5 0.5 1.0 1.0")  # Placeholder bbox (fix later)

        return img, category_label  # Label can later be mapped to class IDs

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform, save_labels=True)
val_dataset = YOLODataset("val", transform, save_labels=True)
test_dataset = YOLODataset("test", transform, save_labels=True)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✅ Dataset now correctly maps images to labels based on folder structure.")


In [ ]:
import torch

# ✅ IoU Calculation in PyTorch
def siou_loss(pred_box, gt_box):
    """
    Compute SIoU loss between predicted and ground truth boxes.
    pred_box, gt_box: Tensor (x_center, y_center, width, height)
    """
    x_p, y_p, w_p, h_p = pred_box[..., 0], pred_box[..., 1], pred_box[..., 2], pred_box[..., 3]
    x_g, y_g, w_g, h_g = gt_box[..., 0], gt_box[..., 1], gt_box[..., 2], gt_box[..., 3]

    # Compute aspect ratio penalty
    aspect_ratio_pred = w_p / h_p
    aspect_ratio_gt = w_g / h_g
    aspect_ratio_loss = torch.abs(torch.log(aspect_ratio_pred / aspect_ratio_gt))

    # Compute IoU
    inter_x1 = torch.max(x_p - w_p / 2, x_g - w_g / 2)
    inter_y1 = torch.max(y_p - h_p / 2, y_g - h_g / 2)
    inter_x2 = torch.min(x_p + w_p / 2, x_g + w_g / 2)
    inter_y2 = torch.min(y_p + h_p / 2, y_g + h_g / 2)

    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)
    union_area = w_p * h_p + w_g * h_g - inter_area
    iou = inter_area / (union_area + 1e-6)

    # Compute SIoU loss (IoU + aspect ratio penalty)
    siou = iou - aspect_ratio_loss

    return 1 - siou  # Loss decreases as SIoU improves

print("✅ IoU function ready for bounding box evaluation.")


In [ ]:
import torch.optim as optim

# ✅ Ensure model is in training mode
enhanced_model.train()
enhanced_model.to(device)

# ✅ Define optimizer & scheduler
optimizer = optim.Adam(enhanced_model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=15, gamma=0.5)  # Learning rate decay

# 🔄 Training loop
for epoch in range(50):  # Adjust epochs as needed
    total_loss = 0

    for batch_idx, (images, labels) in enumerate(train_loader):  # ✅ Use enumerate for batch tracking
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = enhanced_model(images)
        
        # ✅ Ensure YOLO loss is correctly defined
        loss = yolo_loss(outputs, labels)  # Modify if using YOLO11's loss structure
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        if batch_idx % 10 == 0:  # ✅ Print progress every 10 batches
            print(f"Epoch {epoch+1}, Batch {batch_idx}: Loss = {loss.item():.4f}")

    scheduler.step()  # ✅ Adjust learning rate

    print(f"Epoch {epoch+1}: Avg Loss = {total_loss / len(train_loader):.4f}")


I will train bounding boxes later and first just worry about classification -> I dont have annotations which I have to do manually later.

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image
from ultralytics import YOLO

# 📂 Base path for dataset
base_path = r"E:\yolo_project\mined"

# ✅ Ensure class mapping is correctly built from categories (not train/val/test)
class_mapping = {}
for split in ["train", "val", "test"]:
    split_path = os.path.join(base_path, split)
    if not os.path.exists(split_path):
        continue
    for category in os.listdir(split_path):
        category_path = os.path.join(split_path, category)
        if os.path.isdir(category_path):  # ✅ Ensure it's a folder, not a file
            if category not in class_mapping:
                class_mapping[category] = len(class_mapping)

print("✅ Verified Class Mapping:", class_mapping)

# ✅ Define dataset loader
class YOLODataset(Dataset):
    def __init__(self, dataset_split, transform=None):
        self.split_path = os.path.join(base_path, dataset_split)
        self.transform = transform
        self.img_files = []
        self.labels = []

        for category in os.listdir(self.split_path):
            category_path = os.path.join(self.split_path, category)
            if not os.path.isdir(category_path):
                continue
            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(category)  # ✅ Use folder name as label

        print(f"📊 Found {len(self.img_files)} images in {dataset_split}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        category_label = self.labels[idx]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        return img, category_label

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=False)

# ✅ Load YOLO11 backbone for classification
class YOLOClassifier(torch.nn.Module):
    def __init__(self, base_model, num_classes):
        super(YOLOClassifier, self).__init__()
        self.base_model = base_model.model.model[:-1]  # Remove detection head
        self.classifier = torch.nn.Linear(1024, num_classes)  # Classification head

    def forward(self, x):
        print(f"🔍 Input Image Shape: {x.shape}")  # Debug step
        x = self.base_model(x)  # Feature extraction
        print(f"🔍 YOLO Backbone Output Shape: {x.shape}")  # Debug step
        
        x = x.mean([2, 3])  # Global Average Pooling
        print(f"🔍 Shape After Global Pooling: {x.shape}")  # Debug step

        x = x.view(x.size(0), -1)  # Flatten before classification
        print(f"🔍 Shape Before Classifier: {x.shape}")  # Debug step

        x = self.classifier(x)  # Pass through classifier
        return x

# ✅ Load pretrained YOLO backbone
yolo_base = YOLO("yolo11n.pt")
classifier_model = YOLOClassifier(yolo_base, num_classes=len(class_mapping)).to("cuda")

# ✅ Debug Tensor Flow
for images, labels in train_loader:
    images = images.to("cuda")

    # ✅ Convert labels to numerical indices safely
    labels = [class_mapping.get(label, -1) for label in labels]
    labels = [label for label in labels if label != -1]  # Filter out invalid labels
    labels = torch.tensor(labels, dtype=torch.long).to("cuda")

    optimizer = torch.optim.Adam(classifier_model.parameters(), lr=1e-4)
    criterion = torch.nn.CrossEntropyLoss()

    optimizer.zero_grad()
    outputs = classifier_model(images)  # ✅ Debug model flow
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    
    print("✅ Successfully processed first batch!")
    break  # Only test one batch for debugging


Step 1: Extract Features from YOLO’s Backbond \n
Step 2: Train a Separate Classifier \n
Step 3: Use classifier as pre step for future yolo model

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import os
from PIL import Image

# 📂 Updated base path for NVMe storage
base_path = r"C:\yolo_project\mined"  # ✅ Adjusted to match new structure

# ✅ Class Mapping (Ensure consistency)
class_mapping = {
    'construction_crane': 0, 'bulldozer': 1, 'forklift': 2, 'excavator': 3, 'cement_mixer': 4,
    'dump_truck': 5, 'backhoe': 6, 'loader': 7, 'paver': 8, 'hard_hat': 9, 'safety_vest': 10,
    'safety_goggles': 11, 'gloves': 12, 'boots': 13, 'harness': 14, 'respirator_mask': 15,
    'scaffolding': 16, 'barricade': 17, 'traffic_cone': 18, 'construction_sign': 19,
    'wheelbarrow': 20, 'ladder': 21, 'cables_wiring': 22, 'unstable_structure': 23,
    'exposed_wiring': 24, 'falling_debris': 25, 'fire_risk': 26, 'oil_spill': 27
}
num_classes = len(class_mapping)

class YOLODataset(Dataset):
    def __init__(self, dataset_split, transform=None):
        self.split_path = os.path.join(base_path, dataset_split)  # ✅ Ensures correct folder structure
        self.transform = transform

        # 🔄 Collect all images and their class labels
        self.img_files = []
        self.labels = []
        for category in os.listdir(self.split_path):  # ✅ Reads categories inside train/val/test folders
            category_path = os.path.join(self.split_path, category)
            if not os.path.isdir(category_path):
                continue

            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(class_mapping.get(category, -1))  # ✅ Map string labels to class IDs

        print(f"📊 Found {len(self.img_files)} images in {dataset_split}")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        label_idx = self.labels[idx]  # ✅ Class ID instead of string

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # 🔄 Convert class index to multi-label format (One-hot encoding)
        label_tensor = torch.zeros(num_classes)
        label_tensor[label_idx] = 1.0  # ✅ Multi-label format for BCE Loss

        return img, label_tensor  # ✅ Outputs: Image, One-hot encoded label tensor

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((640, 640)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform)
val_dataset = YOLODataset("val", transform)
test_dataset = YOLODataset("test", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print("✅ Dataset now loading from NVMe storage for faster access.")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=640*640*3, hidden_dim=512, num_classes=28):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # Multi-label classification activation

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten input
        x = self.relu(self.fc1(x))
        x = self.sigmoid(self.fc2(x))
        return x  # Multi-label probability output

classifier = MLPClassifier()


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")


In [ ]:
print(torch.cuda.get_device_name(0))  # Prints your GPU name


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class MLPClassifier(nn.Module):
    def __init__(self, input_dim=640*640*3, hidden_dim=512, num_classes=28):
        super(MLPClassifier, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)  # ✅ Batch normalization added
        self.fc2 = nn.Linear(hidden_dim, num_classes)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()  # Multi-label classification activation

        # ✅ Initialize weights for better gradient flow
        torch.nn.init.kaiming_normal_(self.fc1.weight)
        torch.nn.init.zeros_(self.fc1.bias)
        torch.nn.init.xavier_normal_(self.fc2.weight)
        torch.nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)  # Flatten input
        x = self.relu(self.bn1(self.fc1(x)))  # ✅ Batch norm applied
        x = self.sigmoid(self.fc2(x))
        return x  # Multi-label probability output

classifier = MLPClassifier()

# Move classifier to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
classifier.to(device)

# Initialize optimizer
optimizer = optim.Adam(classifier.parameters(), lr=0.0001)
criterion = nn.BCELoss()  # Binary cross-entropy for multi-label classification

# Training loop with enhanced debug prints
for epoch in range(10):
    print(f"\n🚀 Epoch {epoch+1}/{10} Starting...")
    epoch_loss = 0
    
    for batch_idx, (images, labels) in enumerate(train_loader):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = classifier(images)

        loss = criterion(outputs, labels)
        loss.backward()

        # 🔬 Debugging: Print activation range & gradient updates
        print(f"🔬 Activation Stats | Mean: {outputs.mean().item():.4f}, Std: {outputs.std().item():.4f}")
        for name, param in classifier.named_parameters():
            if param.grad is not None:
                print(f"🔄 Gradient check | {name}: {param.grad.abs().mean().item():.6f}")

        optimizer.step()
        epoch_loss += loss.item()

        # 🔍 Periodic Debug Prints
        if batch_idx % 10 == 0:
            print(f"📊 Batch {batch_idx}/{len(train_loader)} | Loss: {loss.item():.4f}")
        
    print(f"✅ Epoch {epoch+1} Completed | Avg Loss: {epoch_loss / len(train_loader):.4f}")

print("🎉 Training Finished!")


In [ ]:
torch.save({
    "epoch": epoch,
    "model_state_dict": classifier.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": epoch_loss
}, r"C:\yolo_project\transfer_learning\mlp_classifier_full.pt")


In [ ]:
import torch
import numpy as np

# ✅ Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load the trained model
checkpoint_path = "mlp_classifier_full.pt"
classifier = MLPClassifier().to(device)
classifier.load_state_dict(torch.load(checkpoint_path)["model_state_dict"])
classifier.eval()  # Set to evaluation mode

print(f"🔄 Loaded model from checkpoint, trained up to Epoch {torch.load(checkpoint_path)['epoch']}")

# ✅ Run batch inference
all_outputs = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = classifier(images)  # ✅ Get predictions for batch
        all_outputs.append(outputs.cpu().numpy())  # Convert to NumPy for analysis
        all_labels.append(labels.cpu().numpy())

# 🔄 Convert lists to arrays for thresholding analysis
all_outputs = np.vstack(all_outputs)
all_labels = np.vstack(all_labels)

# ✅ Apply threshold for classification
threshold = 0.5  # Adjust as needed
predictions = (all_outputs > threshold).astype(int)

# ✅ Print batch results
for i in range(len(predictions)):
    print(f"📊 Image {i+1}: Predicted Labels: {predictions[i]}")


In [ ]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from PIL import Image

# ✅ Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load the trained model
checkpoint_path = r"C:\yolo_project\transfer_learning\mlp_classifier_full.pt"

classifier = MLPClassifier().to(device)
classifier.load_state_dict(torch.load(checkpoint_path)["model_state_dict"])
classifier.eval()  # Set to evaluation mode

print(f"🔄 Loaded model from checkpoint, trained up to Epoch {torch.load(checkpoint_path)['epoch']}")

# ✅ Run batch inference
all_outputs = []
all_labels = []
image_indices = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = classifier(images)  # ✅ Get predictions for batch
        all_outputs.append(outputs.cpu().numpy())  # Convert to NumPy for analysis
        all_labels.append(labels.cpu().numpy())

        # ✅ Store image indices for reference (only first 5 images)
        if len(image_indices) < 5:
            image_indices.extend(list(range(batch_idx * len(images), batch_idx * len(images) + len(images))))
            if len(image_indices) > 5:
                image_indices = image_indices[:5]  # Ensure only 5 images are included

# 🔄 Convert lists to arrays for thresholding analysis
all_outputs = np.vstack(all_outputs)
all_labels = np.vstack(all_labels)

# ✅ Apply threshold for classification
threshold = 0.5  # Adjust as needed
predictions = (all_outputs > threshold).astype(int)

# ✅ Select first 5 images and summarize predictions
print("\n📊 **Sample Predictions from First 5 Images**:")
for idx in image_indices:
    # Load and display the image for the current index
    img_path = test_dataset.img_files[idx]
    img = Image.open(img_path)
    img.show()
    predicted_classes = [class_name for class_name, class_idx in class_mapping.items() if predictions[idx][class_idx] == 1]
    print(f"🖼️ Image {idx+1}: **Flagged as** {', '.join(predicted_classes) if predicted_classes else 'No flags'}")

# ✅ Compute batch evaluation metrics
precision = precision_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)
recall = recall_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)
f1 = f1_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)

torch.save({
    "epoch": epoch,
    "model_state_dict": classifier.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": epoch_loss,
    "precision": precision,  # ✅ Save precision
    "recall": recall,        # ✅ Save recall
    "f1_score": f1           # ✅ Save F1-score
}, r"C:\yolo_project\transfer_learning\mlp_classifier_full.pt")

print(f"💾 Model & Performance Saved for Epoch {epoch}")

print("\n🔎 **Batch Performance Metrics:**")
print(f"✅ **Precision:** {precision:.4f}")
print(f"✅ **Recall:** {recall:.4f}")
print(f"✅ **F1 Score:** {f1:.4f}")


💡 Comprehensive Summary of Optimizations below

✅ ResNet-50 replaces MLP for more efficient feature extraction. ✅ 224×224 input size ensures faster model training and reduced memory usage. ✅ Bootstrap resampling replaces K-Fold Cross Validation for better generalization in small datasets. ✅ Vectorized loss computation improves efficiency by eliminating unnecessary loops. ✅ Reduced epochs (5 instead of 10) to leverage transfer learning effectively. ✅ Mixed Precision (AMP) accelerates GPU training by optimizing memory usage. ✅ Persistent workers in DataLoader enhance concurrency for faster CPU-to-GPU data transfers. ✅ Optimized tensor transfers (non_blocking=True) for asynchronous memory operations. ✅ Asynchronous gradient updates (torch.cuda.synchronize()) prevent training bottlenecks.

In [4]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, SubsetRandomSampler
import os
from PIL import Image
from sklearn.model_selection import train_test_split

# 📂 Updated base path for NVMe storage
base_path = r"C:\yolo_project\mined"

# ✅ Class Mapping (Ensure consistency)
class_mapping = {
    'construction_crane': 0, 'bulldozer': 1, 'forklift': 2, 'excavator': 3, 'cement_mixer': 4,
    'dump_truck': 5, 'backhoe': 6, 'loader': 7, 'paver': 8, 'hard_hat': 9, 'safety_vest': 10,
    'safety_goggles': 11, 'gloves': 12, 'boots': 13, 'harness': 14, 'respirator_mask': 15,
    'scaffolding': 16, 'barricade': 17, 'traffic_cone': 18, 'construction_sign': 19,
    'wheelbarrow': 20, 'ladder': 21, 'cables_wiring': 22, 'unstable_structure': 23,
    'exposed_wiring': 24, 'falling_debris': 25, 'fire_risk': 26, 'oil_spill': 27
}
num_classes = len(class_mapping)

class YOLODataset(Dataset):
    def __init__(self, split, transform=None):
        self.transform = transform
        self.img_files = []
        self.labels = []
        
        split_path = os.path.join(base_path, split)
        if not os.path.exists(split_path):
            raise ValueError(f"❌ Dataset split '{split}' not found at {split_path}")

        # ✅ Collect images and labels from requested split
        for category in os.listdir(split_path):
            category_path = os.path.join(split_path, category)
            if not os.path.isdir(category_path):
                continue
            for img in os.listdir(category_path):
                if img.endswith((".jpg", ".png")):
                    self.img_files.append(os.path.join(category_path, img))
                    self.labels.append(class_mapping.get(category, -1))  # ✅ Map labels

        print(f"📊 Found {len(self.img_files)} images in {split} split")

    def __len__(self):
        return len(self.img_files)

    def __getitem__(self, idx):
        img_path = self.img_files[idx]
        label_idx = self.labels[idx]

        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # 🔄 Convert class index to one-hot format for multi-label classification
        label_tensor = torch.zeros(num_classes)
        label_tensor[label_idx] = 1.0

        return img, label_tensor

# 🔄 Apply transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

# ✅ Create dataset loaders
train_dataset = YOLODataset("train", transform)
val_dataset = YOLODataset("val", transform)
test_dataset = YOLODataset("test", transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=4, pin_memory=True, persistent_workers=False)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=False)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=False)

print("✅ Dataset successfully loaded from NVMe storage!")


📊 Found 2243 images in train split
📊 Found 1334 images in val split
📊 Found 389 images in test split
✅ Dataset successfully loaded from NVMe storage!


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as transforms
from sklearn.model_selection import train_test_split
import time

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResNetClassifier(nn.Module):
    def __init__(self, num_classes=28):
        super(ResNetClassifier, self).__init__()
        self.resnet = models.resnet50(pretrained=True)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        return x

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

num_folds = 5
BATCH_SIZE = 64

print(f"Dataset length: {len(train_dataset)}")

for fold in range(num_folds):
    print(f"\n🚀 Fold {fold+1}/{num_folds} (Bootstrap Resampling) Starting...")

    indices = list(range(len(train_dataset)))
    train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=fold)

    print(f"  ➡️ Training samples: {len(train_idx)} | Validation samples: {len(val_idx)}")
    print(f"  First 10 train_idx: {train_idx[:10]}")
    print(f"  First 10 val_idx: {val_idx[:10]}")
    print(f"  Any overlap? {set(train_idx) & set(val_idx)}")
    print(f"  Max train_idx: {max(train_idx)}, Max val_idx: {max(val_idx)}")
    print(f"  Min train_idx: {min(train_idx)}, Min val_idx: {min(val_idx)}")

    train_sampler = torch.utils.data.SubsetRandomSampler(train_idx)
    val_sampler = torch.utils.data.SubsetRandomSampler(val_idx)

    if fold == 0:
        nw = 0
        pw = False
    else:
        nw = 8
        pw = True

    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=train_sampler,
        num_workers=nw, pin_memory=True, persistent_workers=pw
    )
    val_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=BATCH_SIZE, sampler=val_sampler,
        num_workers=nw, pin_memory=True, persistent_workers=pw
    )

    print("Testing train_loader iteration for 3 batches...")
    try:
        for i, (images, labels) in enumerate(train_loader):
            print(f"  Batch {i}: images shape {images.shape}, labels shape {labels.shape}")
            if i == 2:
                break
        print("train_loader iteration works!")
    except Exception as e:
        print(f"Error during train_loader iteration: {e}")
        break

    classifier = ResNetClassifier().to(device)
    optimizer = optim.Adam(classifier.parameters(), lr=0.0001)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.cuda.amp.GradScaler()

    for epoch in range(5):
        print(f"\n🔥 Epoch {epoch+1}/5 Starting (Fold {fold+1})...")

        classifier.train()
        all_losses = []
        batch_times = []
        for batch_idx, (images, labels) in enumerate(train_loader):
            start_time = time.time()
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast():
                outputs = classifier(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            torch.cuda.synchronize()
            scaler.step(optimizer)
            scaler.update()

            all_losses.append(loss)
            batch_time = time.time() - start_time
            batch_times.append(batch_time)

            if batch_idx % 5 == 0 or batch_idx == len(train_loader) - 1:
                mem = torch.cuda.memory_allocated(device) / 1024**2
                max_mem = torch.cuda.max_memory_allocated(device) / 1024**2
                print(f"    📊 [Train] Batch {batch_idx+1}/{len(train_loader)} | Loss: {loss.item():.4f} | Batch time: {batch_time:.2f}s | GPU mem: {mem:.1f}MB (max {max_mem:.1f}MB)")

        epoch_loss = torch.stack(all_losses).mean().item()
        avg_batch_time = sum(batch_times) / len(batch_times)
        print(f"  ✅ Epoch {epoch+1} Training Finished | Avg Loss: {epoch_loss:.4f} | Avg batch time: {avg_batch_time:.2f}s")

        classifier.eval()
        val_losses = []
        val_batch_times = []
        with torch.no_grad():
            for val_batch_idx, (val_images, val_labels) in enumerate(val_loader):
                start_time = time.time()
                val_images = val_images.to(device, non_blocking=True)
                val_labels = val_labels.to(device, non_blocking=True)
                with torch.cuda.amp.autocast():
                    val_outputs = classifier(val_images)
                    val_loss = criterion(val_outputs, val_labels)
                val_losses.append(val_loss)
                val_batch_time = time.time() - start_time
                val_batch_times.append(val_batch_time)
                if val_batch_idx % 10 == 0 or val_batch_idx == len(val_loader) - 1:
                    mem = torch.cuda.memory_allocated(device) / 1024**2
                    print(f"    🧪 [Val] Batch {val_batch_idx+1}/{len(val_loader)} | Loss: {val_loss.item():.4f} | Batch time: {val_batch_time:.2f}s | GPU mem: {mem:.1f}MB")

        if val_losses:
            avg_val_loss = torch.stack(val_losses).mean().item()
            avg_val_batch_time = sum(val_batch_times) / len(val_batch_times)
            print(f"  🏁 Epoch {epoch+1} Validation Finished | Avg Val Loss: {avg_val_loss:.4f} | Avg batch time: {avg_val_batch_time:.2f}s")
        else:
            print("  ⚠️ No validation samples in this fold.")

        # Save per epoch, per fold
        torch.save({
            "epoch": epoch+1,
            "model_state_dict": classifier.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": epoch_loss
        }, f"C:\\yolo_project\\transfer_learning\\resnet_classifier_fold{fold+1}_epoch{epoch+1}.pt")
        print(f"  💾 Model saved for Fold {fold+1} Epoch {epoch+1}")

    # Save final model for this fold
    torch.save({
        "epoch": epoch+1,
        "model_state_dict": classifier.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "loss": epoch_loss
    }, f"C:\\yolo_project\\transfer_learning\\resnet_classifier_fold{fold+1}_final.pt")
    print(f"  🏁 Final model saved for Fold {fold+1}")

print("🎉 Training + Bootstrap Resampling Finished!")

Dataset length: 2243

🚀 Fold 1/5 (Bootstrap Resampling) Starting...
  ➡️ Training samples: 1794 | Validation samples: 449
  First 10 train_idx: [985, 1944, 654, 704, 1762, 1226, 1874, 2018, 1693, 287]
  First 10 val_idx: [107, 1005, 565, 564, 39, 1869, 2238, 206, 135, 672]
  Any overlap? set()
  Max train_idx: 2242, Max val_idx: 2240
  Min train_idx: 0, Min val_idx: 6
Testing train_loader iteration for 3 batches...
  Batch 0: images shape torch.Size([64, 3, 224, 224]), labels shape torch.Size([64, 28])
  Batch 1: images shape torch.Size([64, 3, 224, 224]), labels shape torch.Size([64, 28])
  Batch 2: images shape torch.Size([64, 3, 224, 224]), labels shape torch.Size([64, 28])
train_loader iteration works!


C:\Users\achyu\AppData\Local\Temp\ipykernel_12204\664637186.py:77: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()



🔥 Epoch 1/5 Starting (Fold 1)...


C:\Users\achyu\AppData\Local\Temp\ipykernel_12204\664637186.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


    📊 [Train] Batch 1/29 | Loss: 0.7470 | Batch time: 0.23s | GPU mem: 420.7MB (max 5916.4MB)
    📊 [Train] Batch 6/29 | Loss: 0.5195 | Batch time: 0.21s | GPU mem: 420.8MB (max 5916.4MB)
    📊 [Train] Batch 11/29 | Loss: 0.3546 | Batch time: 0.20s | GPU mem: 420.8MB (max 5916.4MB)
    📊 [Train] Batch 16/29 | Loss: 0.2454 | Batch time: 0.22s | GPU mem: 420.8MB (max 5916.4MB)
    📊 [Train] Batch 21/29 | Loss: 0.1804 | Batch time: 0.22s | GPU mem: 420.8MB (max 5916.4MB)
    📊 [Train] Batch 26/29 | Loss: 0.1474 | Batch time: 0.21s | GPU mem: 420.8MB (max 5916.4MB)
    📊 [Train] Batch 29/29 | Loss: 0.1625 | Batch time: 0.26s | GPU mem: 381.5MB (max 5916.4MB)
  ✅ Epoch 1 Training Finished | Avg Loss: 0.3269 | Avg batch time: 0.21s


C:\Users\achyu\AppData\Local\Temp\ipykernel_12204\664637186.py:122: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():


    🧪 [Val] Batch 1/8 | Loss: 0.1259 | Batch time: 0.02s | GPU mem: 418.2MB
    🧪 [Val] Batch 8/8 | Loss: 0.0963 | Batch time: 0.07s | GPU mem: 382.0MB
  🏁 Epoch 1 Validation Finished | Avg Val Loss: 0.1242 | Avg batch time: 0.02s
  💾 Model saved for Fold 1 Epoch 1

🔥 Epoch 2/5 Starting (Fold 1)...
    📊 [Train] Batch 1/29 | Loss: 0.1142 | Batch time: 0.22s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 6/29 | Loss: 0.1015 | Batch time: 0.20s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 11/29 | Loss: 0.0880 | Batch time: 0.22s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 16/29 | Loss: 0.0753 | Batch time: 0.22s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 21/29 | Loss: 0.0658 | Batch time: 0.22s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 26/29 | Loss: 0.0563 | Batch time: 0.22s | GPU mem: 421.4MB (max 5916.4MB)
    📊 [Train] Batch 29/29 | Loss: 0.1182 | Batch time: 0.04s | GPU mem: 382.0MB (max 5916.4MB)
  ✅ Epoch 2 Training Finished | Avg L

Threshold analysis for after annotations, testing below

In [ ]:
import torch
import numpy as np
from sklearn.metrics import precision_score, recall_score, f1_score
from PIL import Image

# ✅ Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ Load the trained model
checkpoint_path = r"C:\yolo_project\transfer_learning\resnet_classifier_fold1_final.pt"

import torchvision.models as models
import torch.nn as nn

class ResNetClassifier(nn.Module):
    def __init__(self, num_classes=28):
        super(ResNetClassifier, self).__init__()
        self.resnet = models.resnet50(pretrained=False)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)

    def forward(self, x):
        x = self.resnet(x)
        return x

classifier = ResNetClassifier().to(device)
classifier.load_state_dict(torch.load(checkpoint_path)["model_state_dict"])
classifier.eval()  # Set to evaluation mode

print(f"🔄 Loaded model from checkpoint, trained up to Epoch {torch.load(checkpoint_path)['epoch']}")

# ✅ Run batch inference
all_outputs = []
all_labels = []
image_indices = []

with torch.no_grad():
    for batch_idx, (images, labels) in enumerate(test_loader):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = classifier(images)  # ✅ Get predictions for batch
        all_outputs.append(outputs.cpu().numpy())  # Convert to NumPy for analysis
        all_labels.append(labels.cpu().numpy())

        # ✅ Store image indices for reference (only first 5 images)
        if len(image_indices) < 5:
            image_indices.extend(list(range(batch_idx * len(images), batch_idx * len(images) + len(images))))
            if len(image_indices) > 5:
                image_indices = image_indices[:5]  # Ensure only 5 images are included

# 🔄 Convert lists to arrays for thresholding analysis
all_outputs = np.vstack(all_outputs)
all_labels = np.vstack(all_labels)

# ✅ Apply threshold for classification
threshold = 0.5  # Adjust as needed
predictions = (all_outputs > threshold).astype(int)

# ✅ Select first 5 images and summarize predictions
print("\n📊 **Sample Predictions from First 5 Images**:")
for idx in image_indices:
    # Load and display the image for the current index
    img_path = test_dataset.img_files[idx]
    img = Image.open(img_path)
    img.show()
    predicted_classes = [class_name for class_name, class_idx in class_mapping.items() if predictions[idx][class_idx] == 1]
    print(f"🖼️ Image {idx+1}: **Flagged as** {', '.join(predicted_classes) if predicted_classes else 'No flags'}")

# ✅ Compute batch evaluation metrics
precision = precision_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)
recall = recall_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)
f1 = f1_score(all_labels.flatten(), predictions.flatten(), average="macro", zero_division=0)

torch.save({
    "epoch": epoch,
    "model_state_dict": classifier.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": epoch_loss,
    "precision": precision,  # ✅ Save precision
    "recall": recall,        # ✅ Save recall
    "f1_score": f1           # ✅ Save F1-score
}, r"C:\yolo_project\transfer_learning\resnet_classifier_fold1_final.pt")

print(f"💾 Model & Performance Saved for Epoch {epoch}")

print("\n🔎 **Batch Performance Metrics:**")
print(f"✅ **Precision:** {precision:.4f}")
print(f"✅ **Recall:** {recall:.4f}")
print(f"✅ **F1 Score:** {f1:.4f}")


🔄 Loaded model from checkpoint, trained up to Epoch 5
